<a href="https://colab.research.google.com/github/kaizengrowth/100-days-of-code-challenge/blob/master/AI_Evaluations_Bootcamp_H1_EvaluatingLLMJudgeFaithfulness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Evaluations Bootcamp - Hands-on 1: Evaluating LLM-as-a-Judge faithfulness metrics

In this notebook we will investigate how the LLM-as-a-Judge faithfulness metrics of [DeepEval](https://github.com/confident-ai/deepeval) and [RAGAS](https://github.com/explodinggradients/ragas) handle entailment, neutrality, and contradiction.

For that, we will use the [SNLI benchmark](https://huggingface.co/datasets/stanfordnlp/snli) (Stanford Natural Language Inference), a widely used dataset in natural language understanding research.

Each example in the SNLI dataset consists of two short sentences, a premise and a hypothesis, and a human-provided label that can be one of the following:

* Entailment: the hypothesis can be logically inferred from the premise.

* Neutral: the hypothesis is consistent with the premise but not directly supported by it.

* Contradiction: the hypothesis directly conflicts with the premise

We treat the premise as if it were the retrieved context in a RAG pipeline and the hypothesis as if it were the generated answer





In [ ]:
!pip install -q "ragas==0.2.14" "deepeval==2.4.5" "langchain==0.3.13" "langchain-community==0.3.13" "langchain-core==0.3.28" "langchain-openai==0.2.14" "datasets"


## Step 1: Load SNLI dataset

In [ ]:
from datasets import load_dataset

In [ ]:
# Load SNLI dataset from Hugging Face
dataset = load_dataset("stanfordnlp/snli")
dataset

In [ ]:
# Get and inspect a random subset of the "test" data
filtered_dataset = dataset['test'].filter(lambda x: x["label"] != -1)

evaluation_subset = filtered_dataset.shuffle(seed=42).select(range(100))

for item in evaluation_subset.to_list()[:10]:

  print(f"Premise: {item['premise']}")
  print(f"Hypothesis: {item['hypothesis']}")

  # 1 = neutral, 2 =contradiction, 0 = entailment
  label = "Entailment" if item['label'] == 0 else "Neutral" if item['label'] == 1 else "Contradiction" if item['label'] == 2 else "Unknown"

  print(f"Label: {label}")
  print()


## Step 2: Calculate RAGAS and DeepEval faithfulness scores on the SNLI random subset

In [ ]:
# Load OpenAI API key
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

In [ ]:
# Method to calculate faithfulness with DeepEval
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric

def calculate_deepeval_faithfulness(output, context):

  metric = FaithfulnessMetric(
      threshold=0.0,
      model="gpt-4o",
      include_reason=False,
  )
  test_case = LLMTestCase(
      input="...",
      actual_output=output,
      retrieval_context=context
  )

  metric.measure(test_case)
  return metric.score

In [ ]:
# Method to calculate faithfulness with RAGAS
from langchain_openai import ChatOpenAI
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import Faithfulness
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o", temperature=0)
)

def calculate_ragas_faithfulness(output, context):

  sample = SingleTurnSample(
        user_input="...",
        response=output,
        retrieved_contexts=context
    )
  scorer = Faithfulness(llm=evaluator_llm)
  score = scorer.single_turn_score(sample)
  return score

In [ ]:
# Calculate RAGAS and DeepEval faithfulness metrics on the SNLI random subset and store results into a csv file

import csv
import tqdm

with open('rag_faithfulness_scores.csv', 'w', newline='\n') as csvfile:
    writer = csv.writer(csvfile, delimiter=',')
    writer.writerow(['premise', 'hypothesis','label', 'deepeval_faithfulness', 'ragas_faithfulness'])

    for item in tqdm.tqdm(evaluation_subset.to_list()[:10]):

      deepeval_faithfulness = calculate_deepeval_faithfulness(item['hypothesis'], [item['premise']])
      ragas_faithfulness = calculate_ragas_faithfulness(item['hypothesis'], [item['premise']])
      writer.writerow([item['premise'], item['hypothesis'], item['label'], deepeval_faithfulness, ragas_faithfulness])

      print()

## Step 3: Analyze results

In [ ]:
# Analyze results
import pandas as pd
from sklearn.metrics import precision_score, recall_score
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score, classification_report
import matplotlib.pyplot as plt


def analyze_results(faithfulness_scores_csv, target_label = None):

  # read the data
  df = pd.read_csv(faithfulness_scores_csv)
  df = df.dropna().reset_index(drop=True)

  if target_label == "Entailment":
    df = df[df["label"] == 0]
  elif target_label == "Neutral":
    df = df[df["label"] == 1]
  elif target_label == "Contradiction":
    df = df[df["label"] == 2]


  # map the SNLI labels to the faithfulness scores that each toolkit would be expected to produce
  #  - For entailments, both DeepEval and RAGAS should return 1, since the hypothesis is fully supported by the premise.
  #  - For neutral cases, DeepEval should return 1, since the hypothesis does not contradict the premise, while RAGAS should return 0, since the hypothesis cannot be directly inferred from it.
  #  - For contradictions, Both toolkits should return 0, since the hypothesis is explicitly at odds with the premise.

  # round faithfulness scores to 0 or 1
  df["deepeval_faithfulness"] = (df["deepeval_faithfulness"] >= 0.5).astype(int)
  df["ragas_faithfulness"] = (df["ragas_faithfulness"] >= 0.5).astype(int)

  df["deepeval_truth"] = np.where(df["label"] == 2, 0, 1) # Contradiction label becomes 0, everything else becomes 1
  df["ragas_truth"] = np.where(df["label"] == 0, 1, 0) # Entailment label becomes 1, everything else becomes 0

  # Calculate precision and recall scores

  # # # --- DeepEval ---
  y_true_deep = df["deepeval_truth"]
  y_pred_deep = df["deepeval_faithfulness"]


  print("DeepEval metrics:")
  print(classification_report(y_true_deep, y_pred_deep, digits=3))

  # # # --- RAGAS ---
  y_true_ragas = df["ragas_truth"]
  y_pred_ragas = df["ragas_faithfulness"]


  print("RAGAS metrics:")
  print(classification_report(y_true_ragas, y_pred_ragas, digits=3))




In [ ]:
analyze_results("rag_faithfulness_scores.csv")

In [ ]:
analyze_results("rag_faithfulness_scores.csv","Contradiction")

In [ ]:
analyze_results("rag_faithfulness_scores.csv","Entailment")

In [ ]:
analyze_results("rag_faithfulness_scores.csv","Neutral")